Converts polygon annotations to regular bounding boxes.

In [1]:

from pathlib import Path
import shutil

LABEL_DIR = Path("archive/train/labels")
BACKUP_DIR = Path("archive/train/labels_backup")

if not BACKUP_DIR.exists():
    shutil.copytree(LABEL_DIR, BACKUP_DIR)
    print("Original labels backed up!")

converted = 0
unchanged = 0
invalid = 0

for label_file in LABEL_DIR.glob("*.txt"):

    new_lines = []

    with open(label_file, "r") as f:
        lines = f.readlines()

    for line in lines:

        values = line.strip().split()

        if not values:
            continue

        try:
            class_id = int(values[0])
            coords = list(map(float, values[1:]))

            if len(coords) == 4:
                unchanged += 1
                new_lines.append(
                    f"{class_id} " +
                    " ".join(map(str, coords))
                )
                continue

            if len(coords) < 6 or len(coords) % 2 != 0:
                invalid += 1
                print("Invalid polygon:", label_file.name)
                continue

            xs = coords[0::2]
            ys = coords[1::2]

            if not all(0 <= x <= 1 for x in xs):
                invalid += 1
                continue

            if not all(0 <= y <= 1 for y in ys):
                invalid += 1
                continue

            x_min = min(xs)
            x_max = max(xs)
            y_min = min(ys)
            y_max = max(ys)

            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2

            width = x_max - x_min
            height = y_max - y_min

            new_line = (
                f"{class_id} "
                f"{x_center:.10f} "
                f"{y_center:.10f} "
                f"{width:.10f} "
                f"{height:.10f}"
            )

            new_lines.append(new_line)
            converted += 1

        except (ValueError, IndexError):
            invalid += 1
            print("Could not process:", label_file.name)

    with open(label_file, "w") as f:
        f.write("\n".join(new_lines))

        if new_lines:
            f.write("\n")

print("\n========== SUMMARY ==========")
print("Polygons converted:", converted)
print("Bounding boxes preserved:", unchanged)
print("Invalid annotations:", invalid)
print("Backup location:", BACKUP_DIR)


Original labels backed up!

========== SUMMARY ==========
Polygons converted: 37
Bounding boxes preserved: 868741
Invalid annotations: 0
Backup location: archive\train\labels_backup
